Dataset with parameters : meantemp,meanpressure,humidity,windspeed


In [ ]:
import json
import os
from getpass import getpass

username = input("Enter your Kaggle username: ")
key = getpass("Enter your Kaggle API key: ")

kaggle_creds = {"username": username, "key": key}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("kaggle.json created successfully")

Test if the API key is working fine or not.

In [ ]:
!pip install -q kaggle
!kaggle datasets list -s "daily climate"

Download and unzip the dataset

In [ ]:
!kaggle datasets download -d sumanthvrao/daily-climate-time-series-data
!unzip -o daily-climate-time-series-data.zip

In [ ]:
import pandas as pd

train_df = pd.read_csv('DailyDelhiClimateTrain.csv')
test_df = pd.read_csv('DailyDelhiClimateTest.csv')

print(train_df.shape)
print(train_df.head())
print(train_df.dtypes)
print(train_df.tail())
print(test_df.tail())
print(test_df.head())

Converting the text date data to time series data.

In [ ]:
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

print(train_df.dtypes)

Plot the graph of mean temperature v/s date for simplicity let make date as index.

In [ ]:
import matplotlib.pyplot as plt

train_df= train_df.sort_values('date')
train_df= train_df.set_index('date')

plt.figure(figsize=(15,5))
plt.plot(train_df.index,train_df['meantemp'])
plt.xlabel('Date')
plt.ylabel("Mean Temperature")
plt.title('Delhi Daily Mean Temperature (2013-2017)')
plt.show()

Let's Decompose our data into seasons.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decompose =seasonal_decompose(train_df["meantemp"],model="addictive",period=365)

fig=decompose.plot()
fig.set_size_inches(15,5)
plt.show()

Running the Augmented Dickey-Fuller (ADF) test to check whether meantemp is stationary or not

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(test_df["meantemp"])

print("ADF Statistics : ",result[0])
print("p-value : ",result[1])
print("Critical Values : ")

for key,values in result[4].items():
  print(key,values)

As the p value is > 0.5 lets apply differencing and then run again the ADF test

In [ ]:
# diff.() is pandas built in function it does temp(t)-temp(t-1) i.e differencing
train_df["meantemp_diff"] = train_df["meantemp"].diff()

# dropping the first row as the first row will be null as there is no yesterday for day 1
train_df_diff = train_df.dropna(subset=["meantemp_diff"])

result_diff = adfuller(train_df_diff["meantemp_diff"])

print("ADF Statistic : ", result_diff[0])
print("p-value : ", result_diff[1])
print("Critical Values : ")

for key, values in result_diff[4].items():
    print(key,values)

Let us now find the coorelation between the parameters

In [ ]:
corrmatrix = train_df[["meantemp", "humidity", "wind_speed", "meanpressure"]].corr()
print(corrmatrix)

Before Moving to LSTM let's set a Baseline

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train_df['meantemp'], order=(5,1,0))
modelfit = model.fit()

print(modelfit.summary())

Let's Calculate RMSE of ARIMA model to set it as a baseline for our LSTM

In [ ]:
from sklearn.metrics import mean_squared_error as mse
import numpy as np

forecast = modelfit.forecast(steps=len(test_df))

rmsearima = np.sqrt(mse(test_df["meantemp"], forecast))

print("ARIMA Baseline RMSE : ", rmsearima)

The RMSE came to be very large Let's verify this diagnosis by plotting predicted vs actual

In [ ]:
plt.figure(figsize=(15,5))

plt.title("ARIMA Forecast VS NOWCAST")
plt.plot(test_df.index,test_df["meantemp"],label="Actual")
plt.plot(test_df.index,forecast,label="Predicted")

plt.legend()
plt.show()

Acc to the graph our main issue was that ARIMA model looked for the past 5 days and then forecast the temperature in which it was not able to detect seasonal changes. So what we will do now is that predict for the first day and then check the results for that day and slide away (Nowcasting)

In [ ]:
history = list(train_df["meantemp"])
predictions = []

for t in range(len(test_df)):
    model = ARIMA(history, order=(5,1,0))

    modelfit = model.fit()
    nowcast = modelfit.forecast(steps=1)
    predictedtemp=nowcast[0]
    predictions.append(predictedtemp)
    #Finds the mean temp at index t and append it in history
    history.append(test_df["meantemp"].iloc[t])

new_arima_rmse = np.sqrt(mse(test_df["meantemp"], predictions))
print("Ner ARIMA RMSE : ", new_arima_rmse)

Let's Plot the graph to check how our predictions evnt Just for fun

In [ ]:
plt.figure(figsize=(15,5))
plt.title("ARIMA Rolling (1-day-ahead) Forecast vs Actual")

plt.plot(test_df.index,test_df["meantemp"],label="Actual")
plt.plot(test_df.index,predictions,label="Predicted")

plt.xlabel("Date")
plt.ylabel("Mean Temp")

plt.legend()
plt.show()

Let's split validation data from train data

In [ ]:
parameters=["meantemp","humidity","wind_speed"]

trainsize=int(len(train_df)*0.85)

newtrain_df=train_df.iloc[:trainsize]

val_df=train_df.iloc[trainsize:]

print(newtrain_df.shape, val_df.shape)

Let's now Scale down our data.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(newtrain_df[parameters])

scaled_val = scaler.transform(val_df[parameters])

scaled_test = scaler.transform(test_df[parameters])

print(scaled_train.shape, scaled_val.shape, scaled_test.shape)

Let's perform sequence framing

In [ ]:
import numpy as np

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length, 0])
    return np.array(X), np.array(y)

seq_len = 29

X_train, y_train = create_sequences(scaled_train, seq_len)
X_val, y_val = create_sequences(scaled_val, seq_len)
X_test, y_test = create_sequences(scaled_test, seq_len)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

Let's now start building LSTM model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")

model.summary()

Let's Start the training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop]
)

Let's plot the graph

In [ ]:
plt.figure(figsize=(15,5))

plt.title("Training vs Validation Loss")

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")

plt.legend()
plt.show()

Let's now predict the scaled temperature

In [ ]:
scaled_predictions = model.predict(X_test)

print(scaled_predictions[:5])

print(scaled_predictions.shape)

let's make the predictions in degree celsius by upscaling our predictions

In [ ]:
dummy = np.zeros((len(scaled_predictions), len(parameters)))
dummy[:, 0] = scaled_predictions[:, 0]

actual_predictions = scaler.inverse_transform(dummy)[:, 0]

print(actual_predictions[:5])

In [ ]:
dummy_y = np.zeros((len(y_test), len(parameters)))
dummy_y[:, 0] = y_test

actual_y_test = scaler.inverse_transform(dummy_y)[:, 0]

print(actual_y_test[:5])

RMSE of our LSTM

In [ ]:
rmse_lstm = np.sqrt(mse(actual_y_test, actual_predictions))

print("LSTM RMSE : ", rmse_lstm)

Let's plot all the graphs

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(actual_y_test, label="Actual")
plt.plot(actual_predictions, label="LSTM Prediction")

plt.xlabel('Days into Test Period')
plt.ylabel('Mean Temperature')

plt.title('LSTM Prediction vs Actual')

plt.legend()
plt.show()

In [ ]:
arima_aligned = predictions[seq_len:]  # skiped first 29 to match LSTM's starting point
actual_aligned = test_df['meantemp'].iloc[seq_len:]

plt.figure(figsize=(15,5))

plt.plot(actual_aligned.values, label='Actual')
plt.plot(arima_aligned, label='ARIMA Rolling')
plt.plot(actual_predictions, label='LSTM')

plt.xlabel('Days into Test Period')
plt.ylabel('Mean Temperature')

plt.title('ARIMA vs LSTM vs Actual')

plt.legend()
plt.show()